# Phase 2: Model C - BERTopic Semantic Discovery

This notebook focuses on training the **BERTopic** model to discover semantic topic clusters within the news corpus. This approach goes beyond the probabilistic LDA model by leveraging contextual embeddings (BERT).

### Goals:
1. Train BERTopic on processed news text.
2. Extract document-topic distributions.
3. Save the topic proportions for temporal forecasting in Phase 3.

In [ ]:
import pandas as pd
import numpy as np
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
import os

# 1. Load Data
DATA_PATH = '../data/processed/featured_news.parquet'
df = pd.read_parquet(DATA_PATH)
print(f"Loaded {len(df)} documents.")

# Handle potential NaN values
df['clean_text'] = df['clean_text'].fillna("")
docs = df['clean_text'].tolist()

In [ ]:
# 2. Component Configuration
# UMAP for dimensionality reduction
umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', random_state=42)

# HDBSCAN for clustering
hdbscan_model = HDBSCAN(min_cluster_size=10, metric='euclidean', cluster_selection_method='eom', prediction_data=True)

# Sentence Transformer for embeddings
sentence_model = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
# 3. Train BERTopic Model
# We limit the number of topics to around 20-30 for better forecasting interpretability
topic_model = BERTopic(
    embedding_model=sentence_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    nr_topics=30,
    calculate_probabilities=True,
    verbose=True
)

topics, probs = topic_model.fit_transform(docs)
print("Training completed.")

In [ ]:
# 4. Analyze Topics
topic_info = topic_model.get_topic_info()
print(topic_info.head(10))

# Store topic results in dataframe
df['topic'] = topics
# Store the probability matrix for the forecasting phase
np.save('../data/processed/topic_probabilities.npy', probs)
df.to_parquet('../data/processed/featured_news_with_topics.parquet', index=False)

print("Results saved to data/processed/")